# LeetCode 370 — Range Addition bằng Python

Link đề: https://leetcode.com/problems/range-addition/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Prefix Sum (tổng tiền tố), Difference Array (mảng hiệu)

Skeleton LeetCode:

```python
class Solution:
    def getModifiedArray(self, length: int, updates: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: đây là bài **Premium** nên đề và các ví dụ được viết theo hiểu biết về bài; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu bạn có tài khoản Premium. Ví dụ nào do mình tự dựng sẽ được ghi chú "đối chiếu lại đề gốc".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có mảng `arr` dài `length`, toàn số `0`; mỗi phép cập nhật `[start, end, inc]` **cộng `inc` vào mọi phần tử từ vị trí `start` đến vị trí `end` (gồm cả hai đầu)**; phải trả về mảng sau khi áp dụng **tất cả** các phép cập nhật.
2. Nắm các công cụ Python: `[0] * n` tạo mảng, **cắt lát (slicing)** và vòng `for` lồng nhau, `itertools.accumulate` (tổng tiền tố), `dict` đếm sự kiện, `sorted`, và mẹo bit `i & -i` cho cây Fenwick.
3. Đi từ cách **cộng trực tiếp từng phần tử** → **mảng hiệu (difference array) + tổng tiền tố** → **quét theo các mốc sự kiện (sweep line)** → **cây Fenwick (Binary Indexed Tree)**.
4. Hiểu mấu chốt: thay vì cộng `inc` vào cả đoạn (`O(độ dài đoạn)`), ta chỉ **đánh dấu hai đầu** — `+inc` tại `start` và `-inc` tại `end + 1` — rồi **lấy tổng tiền tố một lần** để "lan" các dấu đó ra toàn đoạn.
5. Tránh các lỗi kinh điển: **quên `end + 1`** (trừ `inc` tại `end`), **tràn chỉ số** khi `end + 1 == length`, **quên lấy tổng tiền tố**, và **cắt lát lệch một** (`arr[start:end]` bỏ sót phần tử `end`).

## 2. Đề bài & ví dụ

Cho số nguyên `length` và mảng `updates`, trong đó `updates[i] = [start_i, end_i, inc_i]`.

Bạn có mảng `arr` gồm `length` phần tử, ban đầu **tất cả bằng `0`**. Ở phép cập nhật thứ `i`, ta **cộng `inc_i`** vào các phần tử `arr[start_i], arr[start_i + 1], ..., arr[end_i]` (cả hai đầu **đều được tính**).

Trả về `arr` sau khi áp dụng **toàn bộ** `updates`.

**Ví dụ 1**

- Input: `length = 5`, `updates = [[1,3,2],[2,4,3],[0,2,-2]]`
- Output: `[-2,0,3,5,3]`
- Giải thích: bắt đầu `[0,0,0,0,0]`; sau `[1,3,2]` được `[0,2,2,2,0]`; sau `[2,4,3]` được `[0,2,5,5,3]`; sau `[0,2,-2]` (chỉ phủ chỉ số `0..2`) được `[-2,0,3,5,3]`.

**Ví dụ 2**

- Input: `length = 10`, `updates = [[2,4,6],[5,6,8],[1,9,-4]]`
- Output: `[0,-4,2,2,2,4,4,-4,-4,-4]`

**Ví dụ tự dựng (đối chiếu lại đề gốc)**

- `length = 4`, `updates = []` → `[0,0,0,0]` (không có phép nào).
- `length = 1`, `updates = [[0,0,5]]` → `[5]`.

**Ràng buộc (theo hiểu biết về đề)**

- `1 <= length <= 10^5`
- `0 <= updates.length <= 10^4`
- `0 <= start_i <= end_i < length`
- `-1000 <= inc_i <= 1000`

Chú ý: với `length` tới `10^5` và tới `10^4` phép cập nhật, cách cộng từng phần tử có thể tốn tới `10^9` bước — quá chậm. Ta cần cách tốt hơn.

In [1]:
from typing import List
import random
from itertools import accumulate

examples = [
    (5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]], [-2, 0, 3, 5, 3]),
    (10, [[2, 4, 6], [5, 6, 8], [1, 9, -4]], [0, -4, 2, 2, 2, 4, 4, -4, -4, -4]),
    (4, [], [0, 0, 0, 0]),
    (1, [[0, 0, 5]], [5]),
]
for length, updates, expected in examples:
    print(f"length={length}, updates={updates} -> kỳ vọng {expected}")

length=5, updates=[[1, 3, 2], [2, 4, 3], [0, 2, -2]] -> kỳ vọng [-2, 0, 3, 5, 3]
length=10, updates=[[2, 4, 6], [5, 6, 8], [1, 9, -4]] -> kỳ vọng [0, -4, 2, 2, 2, 4, 4, -4, -4, -4]
length=4, updates=[] -> kỳ vọng [0, 0, 0, 0]
length=1, updates=[[0, 0, 5]] -> kỳ vọng [5]


Hãy thử "bằng tay" ví dụ 1 để chắc chắn ta hiểu đề: áp dụng từng phép một và in mảng sau mỗi phép.

In [2]:
length, updates = 5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]
arr = [0] * length
print("ban đầu:", arr)
for start, end, inc in updates:
    for i in range(start, end + 1):        # end + 1 vì range không lấy điểm cuối
        arr[i] += inc
    print(f"sau [{start},{end},{inc:+d}]:", arr)

ban đầu: [0, 0, 0, 0, 0]
sau [1,3,+2]: [0, 2, 2, 2, 0]
sau [2,4,+3]: [0, 2, 5, 5, 3]
sau [0,2,-2]: [-2, 0, 3, 5, 3]


Sau ba phép ta được `[-2, 0, 3, 5, 3]`, khớp đề. Chú ý phép cuối `[0,2,-2]` chỉ phủ chỉ số `0..2`, nên phần tử ở chỉ số `3` giữ nguyên `5`. Thói quen tốt: luôn **chạy code để kiểm chứng** thay vì tin vào phép tính nhẩm.

## 3. Kiến thức nền

### 3.1. Tạo mảng toàn số 0 và đọc/ghi chỉ số

`[0] * n` tạo list `n` phần tử `0`. Chỉ số chạy `0 .. n-1`; `arr[i] += v` cộng `v` vào phần tử `i`. Truy cập `arr[n]` gây `IndexError`. Điều này quan trọng vì sau này ta sẽ đánh dấu ở vị trí `end + 1`, có thể bằng đúng `n`.

In [3]:
arr = [0] * 5
arr[2] += 7
arr[4] -= 3
print(arr)                      # [0, 0, 7, 0, -3]
try:
    arr[5] += 1                 # chỉ số 5 không tồn tại (list có 5 phần tử: 0..4)
except IndexError as e:
    print("IndexError:", e)

[0, 0, 7, 0, -3]
IndexError: list index out of range


### 3.2. Đoạn **đóng** `[start, end]` và `range(start, end + 1)`

Đề tính **cả hai đầu**: đoạn `[2, 4]` gồm chỉ số `2, 3, 4` — ba phần tử, tức `end - start + 1`. `range(a, b)` lại **không** lấy `b`, nên để duyệt đoạn đóng phải viết `range(start, end + 1)`. Quên `+ 1` là lỗi lệch một (off-by-one) kinh điển.

In [4]:
start, end = 2, 4
print(list(range(start, end)))          # [2, 3]     thiếu phần tử end
print(list(range(start, end + 1)))      # [2, 3, 4]  đúng với đoạn đóng
print("số phần tử:", end - start + 1)

[2, 3]
[2, 3, 4]
số phần tử: 3


### 3.3. Tổng tiền tố (prefix sum) và `itertools.accumulate`

**Tổng tiền tố** của dãy `a` là dãy `p` với `p[i] = a[0] + a[1] + ... + a[i]`. Cách dựng: `p[0] = a[0]`, `p[i] = p[i-1] + a[i]`. Python có sẵn `itertools.accumulate(a)` làm đúng việc này (trả về iterator, nên bọc `list(...)` để xem).

In [5]:
a = [2, 0, 3, -1, 4]
p = []
running = 0
for x in a:
    running += x
    p.append(running)
print(p)                               # [2, 2, 5, 4, 8]
print(list(accumulate(a)))             # [2, 2, 5, 4, 8]  giống hệt

[2, 2, 5, 4, 8]
[2, 2, 5, 4, 8]


### 3.4. Mảng hiệu (difference array): phép "ngược" của tổng tiền tố

Với mảng `arr`, **mảng hiệu** `diff` được định nghĩa: `diff[0] = arr[0]`, `diff[i] = arr[i] - arr[i-1]`. Nó là phép ngược của tổng tiền tố: **tổng tiền tố của `diff` cho lại `arr`**.

Điều kỳ diệu: nếu muốn **cộng `inc` vào cả đoạn `[l, r]` của `arr`**, trong `diff` ta chỉ phải sửa **hai ô**: `diff[l] += inc` và `diff[r+1] -= inc`. Lý do: sau khi lấy tổng tiền tố, `+inc` ở `l` được cộng vào mọi vị trí từ `l` về sau, còn `-inc` ở `r+1` **triệt tiêu** nó cho mọi vị trí từ `r+1` về sau; kết quả chỉ đoạn `[l, r]` được cộng `inc`.

In [6]:
arr = [0, 2, 5, 5, 3]
diff = [arr[0]] + [arr[i] - arr[i - 1] for i in range(1, len(arr))]
print("diff =", diff)                          # [0, 2, 3, 0, -2]
print("accumulate(diff) =", list(accumulate(diff)))   # [0, 2, 5, 5, 3] = arr

# Cộng 10 vào đoạn [1, 3] chỉ bằng hai thao tác trên diff
diff[1] += 10
diff[3 + 1] -= 10
print("arr sau khi cộng 10 vào [1,3]:", list(accumulate(diff)))   # [0, 12, 15, 15, 3]

diff = [0, 2, 3, 0, -2]
accumulate(diff) = [0, 2, 5, 5, 3]
arr sau khi cộng 10 vào [1,3]: [0, 12, 15, 15, 3]


### 3.5. `dict`, `sorted` và đếm theo vị trí

Khi `length` rất lớn nhưng số cập nhật ít, ta có thể không dùng mảng dài mà chỉ giữ `dict` các **vị trí có thay đổi** (`vị trí -> tổng thay đổi`). `d.get(k, 0)` trả `0` nếu thiếu khóa; `sorted(d)` trả danh sách khóa theo thứ tự tăng.

In [7]:
events = {}
for pos, delta in [(1, 2), (4, -2), (2, 3), (5, -3), (1, 5)]:
    events[pos] = events.get(pos, 0) + delta
print(events)                    # {1: 7, 4: -2, 2: 3, 5: -3}
print(sorted(events))            # [1, 2, 4, 5]
for pos in sorted(events):
    print(pos, "->", events[pos])

{1: 7, 4: -2, 2: 3, 5: -3}
[1, 2, 4, 5]
1 -> 7
2 -> 3
4 -> -2
5 -> -3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ta lấy ví dụ 1: `length = 5`, `updates = [[1,3,2],[2,4,3],[0,2,-2]]`. Vẽ mỗi phép như một "dải" cộng vào các cột:

| Chỉ số | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| `[1,3,2]` |  | +2 | +2 | +2 |  |
| `[2,4,3]` |  |  | +3 | +3 | +3 |
| `[0,2,-2]` | -2 | -2 | -2 |  |  |
| **Tổng** | -2 | 0 | 3 | 5 | 3 |

Mỗi cột chính là **tổng các `inc` của những phép có đoạn phủ cột đó**. Nhìn theo hướng từ trái sang phải, giá trị chỉ **thay đổi tại hai loại điểm**: nơi một dải **bắt đầu** (cộng thêm `inc`) và nơi một dải **vừa kết thúc** (ngay sau `end`, trừ đi `inc`).

Ghi lại các thay đổi đó:

| Vị trí | thay đổi |
|---|---|
| 0 | `-2` (dải 3 bắt đầu) |
| 1 | `+2` (dải 1 bắt đầu) |
| 2 | `+3` (dải 2 bắt đầu) |
| 3 | `+2` (dải 3 kết thúc ở 2, tại 3 trừ đi `-2` nên được `+2`) |
| 4 | `-2` (dải 1 kết thúc ở 3 nên tại 4 trừ `2`) |
| 5 | `-3` (dải 2 kết thúc ở 4, tại 5 trừ `3`; ngoài mảng, bỏ qua) |

Rồi cộng dồn từ trái sang phải: `-2`, `-2+2 = 0`, `0+3 = 3`, `3+2 = 5`, `5-2 = 3` ⇒ `[-2, 0, 3, 5, 3]`. Đúng!

In [8]:
# Tính tay bằng code: ghi các thay đổi rồi cộng dồn
length, updates = 5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]
change = [0] * (length + 1)                # thêm 1 ô để chứa vị trí end + 1 = length
for start, end, inc in updates:
    change[start] += inc                   # dải bắt đầu
    change[end + 1] -= inc                 # dải vừa kết thúc
print("thay đổi theo vị trí:", change)     # [-2, 2, 3, 2, -2, -3]
print("cộng dồn:", list(accumulate(change))[:length])

thay đổi theo vị trí: [-2, 2, 3, 2, -2, -3]
cộng dồn: [-2, 0, 3, 5, 3]


## 5. Cách 1 — Brute Force / baseline: cộng trực tiếp từng phần tử

**Ý tưởng.** Làm đúng như đề: với mỗi phép cập nhật `[start, end, inc]`, dùng vòng `for i in range(start, end + 1)` để cộng `inc` vào từng phần tử. Dễ nhất, bám sát mô tả.

In [9]:
def range_add_brute(length, updates):
    arr = [0] * length
    for start, end, inc in updates:
        for i in range(start, end + 1):        # cả hai đầu đều được cộng
            arr[i] += inc
    return arr


for length, updates, expected in examples:
    print(range_add_brute(length, updates), "(kỳ vọng", expected, ")")

[-2, 0, 3, 5, 3] (kỳ vọng [-2, 0, 3, 5, 3] )
[0, -4, 2, 2, 2, 4, 4, -4, -4, -4] (kỳ vọng [0, -4, 2, 2, 2, 4, 4, -4, -4, -4] )
[0, 0, 0, 0] (kỳ vọng [0, 0, 0, 0] )
[5] (kỳ vọng [5] )


**Độ phức tạp (Cách 1).** Gọi `n = length`, `m = len(updates)`. Mỗi phép tốn tới `n` bước ⇒ **Time `O(n·m)`** trong trường hợp xấu (`10^5 × 10^4 = 10^9` — quá chậm), **Space `O(1)`** phụ (ngoài mảng kết quả).

**Nút thắt:** ta cộng `inc` vào **từng** phần tử của đoạn, trong khi tất cả phần tử trong đoạn nhận **cùng một** `inc`. Có thể ghi nhận "đoạn này nhận `inc`" chỉ bằng **hai** thao tác thay vì cả đoạn không?

## 6. Ý tưởng thuật toán / bước đột phá

**Chỉ đánh dấu hai đầu, lan ra bằng tổng tiền tố.**

- Với mỗi `[start, end, inc]`: `diff[start] += inc` và `diff[end + 1] -= inc`. Mỗi phép tốn đúng `O(1)`.
- Sau khi xử lý hết các phép, kết quả `arr` là **tổng tiền tố của `diff`**: `arr[i] = diff[0] + ... + diff[i]`.

Vì sao hợp lý? Tổng tiền tố "đẩy" mọi `+inc` ở `start` sang phải mãi mãi, và `-inc` ở `end + 1` chặn nó lại đúng sau `end`. Nhờ vậy **mỗi phép chỉ tốn `O(1)`**, cộng một lượt tổng tiền tố `O(n)` ở cuối.

Tổng thời gian: `O(n + m)` — với `n = 10^5`, `m = 10^4` chỉ cỡ `10^5` bước.

Lưu ý chỉ số `end + 1` có thể bằng `length`. Có hai cách xử lý: **cấp `length + 1` ô** cho `diff` (rồi bỏ ô cuối), hoặc **chỉ trừ khi `end + 1 < length`**.

In [10]:
# Mini demo: chỉ hai thao tác cho mỗi phép, rồi accumulate
length, updates = 5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]
diff = [0] * (length + 1)
for start, end, inc in updates:
    diff[start] += inc
    diff[end + 1] -= inc
    print(f"sau [{start},{end},{inc:+d}]: diff = {diff}")
print("kết quả:", list(accumulate(diff))[:length])

sau [1,3,+2]: diff = [0, 2, 0, 0, -2, 0]
sau [2,4,+3]: diff = [0, 2, 3, 0, -2, -3]
sau [0,2,-2]: diff = [-2, 2, 3, 2, -2, -3]
kết quả: [-2, 0, 3, 5, 3]


## 7. Các cách tiếp theo

Ngoài Cách 1 (cộng trực tiếp), ta có thêm **3 cách khác nhau về ý tưởng**: Cách 2 (mảng hiệu + tổng tiền tố), Cách 3 (quét theo các mốc sự kiện, sweep line), Cách 4 (cây Fenwick, cập nhật đoạn - truy vấn điểm).

### Cách 2 — Mảng hiệu (difference array) + tổng tiền tố

**Giải thích từng đoạn.**

1. `diff = [0] * (length + 1)` — thêm **một ô dự phòng** để ghi `diff[end + 1]` khi `end = length - 1`.
2. Với mỗi `[start, end, inc]`: `diff[start] += inc`, `diff[end + 1] -= inc`.
3. Cộng dồn: `run = 0`; với `i = 0..length-1` thực hiện `run += diff[i]`, `res[i] = run`.
4. Ô thừa `diff[length]` không cần cộng dồn (nó nằm ngoài mảng).

In [11]:
def range_add_diff(length, updates):
    diff = [0] * (length + 1)              # ô cuối chứa end + 1 == length
    for start, end, inc in updates:
        diff[start] += inc                 # từ start trở đi cộng thêm inc
        diff[end + 1] -= inc               # từ end + 1 trở đi trừ lại inc
    res = []
    running = 0
    for i in range(length):
        running += diff[i]                 # tổng tiền tố của diff
        res.append(running)
    return res


for length, updates, expected in examples:
    print(range_add_diff(length, updates), "(kỳ vọng", expected, ")")

[-2, 0, 3, 5, 3] (kỳ vọng [-2, 0, 3, 5, 3] )
[0, -4, 2, 2, 2, 4, 4, -4, -4, -4] (kỳ vọng [0, -4, 2, 2, 2, 4, 4, -4, -4, -4] )
[0, 0, 0, 0] (kỳ vọng [0, 0, 0, 0] )
[5] (kỳ vọng [5] )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionDiff:
    def getModifiedArray(self, length: int, updates: List[List[int]]) -> List[int]:
        diff = [0] * (length + 1)
        for start, end, inc in updates:
            diff[start] += inc
            diff[end + 1] -= inc
        return list(accumulate(diff[:length]))


print(SolutionDiff().getModifiedArray(5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]))

[-2, 0, 3, 5, 3]


**Độ phức tạp Cách 2:** **Time `O(n + m)`**, **Space `O(n)`** (mảng `diff` và mảng kết quả).

### Cách 3 — Quét theo các mốc sự kiện (sweep line)

**Giải thích từng đoạn.**

1. Mỗi phép tạo **hai sự kiện**: tại `start` thay đổi `+inc`, tại `end + 1` thay đổi `-inc`. Gộp vào `dict` `events` (nhiều sự kiện cùng vị trí cộng lại).
2. **Sắp xếp** các vị trí có sự kiện. Giữa hai mốc liên tiếp, giá trị **không đổi**.
3. Duyệt các mốc theo thứ tự: với mốc `pos`, **điền đoạn `[prev, pos)` bằng `cur`** (giá trị hiện tại), rồi `cur += events[pos]`, `prev = pos`.
4. Sau mốc cuối, điền nốt `[prev, length)` bằng `cur`.

Điểm khác Cách 2: ta không lan dần từng ô theo tổng tiền tố trên mảng dài, mà **nhảy từ mốc tới mốc** và điền cả đoạn bằng một lát cắt (slice). Cách này **rất hợp** khi tọa độ rất lớn nhưng số phép ít (ví dụ bài **Car Pooling**, hay đếm số người đang có mặt trên trục thời gian).

In [13]:
def range_add_sweep(length, updates):
    events = {}
    for start, end, inc in updates:
        events[start] = events.get(start, 0) + inc
        events[end + 1] = events.get(end + 1, 0) - inc
    res = [0] * length
    cur = 0                                   # giá trị đang áp dụng cho đoạn hiện tại
    prev = 0                                  # đầu đoạn hiện tại
    for pos in sorted(events):
        stop = min(pos, length)               # mốc length (từ end + 1) nằm ngoài mảng
        if stop > prev:
            res[prev:stop] = [cur] * (stop - prev)   # điền cả đoạn [prev, stop) bằng cur
            prev = stop
        cur += events[pos]
    if prev < length:
        res[prev:length] = [cur] * (length - prev)
    return res


for length, updates, expected in examples:
    print(range_add_sweep(length, updates), "(kỳ vọng", expected, ")")

[-2, 0, 3, 5, 3] (kỳ vọng [-2, 0, 3, 5, 3] )
[0, -4, 2, 2, 2, 4, 4, -4, -4, -4] (kỳ vọng [0, -4, 2, 2, 2, 4, 4, -4, -4, -4] )
[0, 0, 0, 0] (kỳ vọng [0, 0, 0, 0] )
[5] (kỳ vọng [5] )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionSweep:
    def getModifiedArray(self, length: int, updates: List[List[int]]) -> List[int]:
        events = {}
        for start, end, inc in updates:
            events[start] = events.get(start, 0) + inc
            events[end + 1] = events.get(end + 1, 0) - inc
        res = [0] * length
        cur = prev = 0
        for pos in sorted(events):
            stop = min(pos, length)
            if stop > prev:
                res[prev:stop] = [cur] * (stop - prev)
                prev = stop
            cur += events[pos]
        if prev < length:
            res[prev:] = [cur] * (length - prev)
        return res


print(SolutionSweep().getModifiedArray(10, [[2, 4, 6], [5, 6, 8], [1, 9, -4]]))

[0, -4, 2, 2, 2, 4, 4, -4, -4, -4]


**Độ phức tạp Cách 3:** gọi `k` là số mốc khác nhau (`k <= 2m`). Sắp xếp `O(k log k)`, điền mảng `O(n)` ⇒ **Time `O(m log m + n)`**, **Space `O(m + n)`** (kết quả cần `O(n)`; nếu chỉ cần danh sách các đoạn hằng số thì bộ nhớ chỉ `O(m)`).

### Cách 4 — Cây Fenwick (Binary Indexed Tree): cập nhật đoạn, truy vấn điểm

**Giải thích từng đoạn.**

1. **Cây Fenwick** lưu một mảng `tree` cho phép làm hai việc đều trong `O(log n)`: `add(i, v)` (cộng `v` vào **ô `i`** của mảng hiệu) và `prefix(i)` (tổng các ô `0..i` của mảng hiệu).
2. Vì giá trị `arr[i]` **đúng bằng tổng tiền tố** của mảng hiệu tại `i`, nên `arr[i] = prefix(i)`.
3. Mỗi phép `[start, end, inc]`: `add(start, +inc)` và `add(end + 1, -inc)`. Cuối cùng `res[i] = prefix(i)`.
4. Mẹo `i & -i`: lấy **bit 1 thấp nhất** của `i`. `i += i & -i` nhảy lên ô "cha" khi cập nhật, `i -= i & -i` nhảy xuống ô cần cộng khi truy vấn. Chỉ số trong cây tính từ `1`.

Cách này chậm hơn Cách 2 (`O(n log n)`) nhưng **hữu ích khi vừa cập nhật đoạn vừa truy vấn xen kẽ** (ta chưa biết hết các phép trước). Với bài này (biết hết trước) Cách 2 tốt hơn; ta dùng Fenwick để thấy cùng một ý "mảng hiệu" qua cấu trúc dữ liệu khác.

In [15]:
# Khám phá i & -i: bit 1 thấp nhất
for i in [1, 2, 3, 4, 6, 8, 12]:
    print(f"i={i:2d} ({i:04b})  i & -i = {i & -i}")

i= 1 (0001)  i & -i = 1
i= 2 (0010)  i & -i = 2
i= 3 (0011)  i & -i = 1
i= 4 (0100)  i & -i = 4
i= 6 (0110)  i & -i = 2
i= 8 (1000)  i & -i = 8
i=12 (1100)  i & -i = 4


In [16]:
def range_add_fenwick(length, updates):
    size = length + 1                          # chỉ số ô tính từ 1; end + 1 tối đa length -> ô length + 1
    tree = [0] * (size + 1)

    def add(i, v):                             # cộng v vào ô i (0-indexed) của mảng hiệu
        i += 1
        while i <= size:
            tree[i] += v
            i += i & -i                        # nhảy lên ô cha

    def prefix(i):                             # tổng các ô 0..i của mảng hiệu
        i += 1
        total = 0
        while i > 0:
            total += tree[i]
            i -= i & -i                        # nhảy xuống
        return total

    for start, end, inc in updates:
        add(start, inc)
        add(end + 1, -inc)
    return [prefix(i) for i in range(length)]


for length, updates, expected in examples:
    print(range_add_fenwick(length, updates), "(kỳ vọng", expected, ")")

[-2, 0, 3, 5, 3] (kỳ vọng [-2, 0, 3, 5, 3] )
[0, -4, 2, 2, 2, 4, 4, -4, -4, -4] (kỳ vọng [0, -4, 2, 2, 2, 4, 4, -4, -4, -4] )
[0, 0, 0, 0] (kỳ vọng [0, 0, 0, 0] )
[5] (kỳ vọng [5] )


In [17]:
# Code nộp LeetCode cho Cách 4
class SolutionFenwick:
    def getModifiedArray(self, length: int, updates: List[List[int]]) -> List[int]:
        size = length + 1
        tree = [0] * (size + 1)

        def add(i, v):
            i += 1
            while i <= size:
                tree[i] += v
                i += i & -i

        def prefix(i):
            i += 1
            total = 0
            while i > 0:
                total += tree[i]
                i -= i & -i
            return total

        for start, end, inc in updates:
            add(start, inc)
            add(end + 1, -inc)
        return [prefix(i) for i in range(length)]


print(SolutionFenwick().getModifiedArray(5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]))

[-2, 0, 3, 5, 3]


**Độ phức tạp Cách 4:** **Time `O((m + n) log n)`**, **Space `O(n)`**.

## 8. TRACE từng bước

Ta trace **Cách 2** (mảng hiệu) trên ví dụ 1, và **Cách 3** (sweep line) trên `length = 8`, `updates = [[1,3,5],[2,5,-2],[6,7,4]]`.

Bảng trace Cách 2 với `length = 5`, `updates = [[1,3,2],[2,4,3],[0,2,-2]]` (mảng `diff` có `length + 1 = 6` ô):

| Sau phép | `diff` |
|---|---|
| ban đầu | `[0, 0, 0, 0, 0, 0]` |
| `[1,3,2]` | `[0, 2, 0, 0, -2, 0]` |
| `[2,4,3]` | `[0, 2, 3, 0, -2, -3]` |
| `[0,2,-2]` | `[-2, 2, 3, 2, -2, -3]` |

Cộng dồn `diff[0..4]`: `-2`, `0`, `3`, `5`, `3`.

In [18]:
def trace_diff(length, updates):
    diff = [0] * (length + 1)
    print("ban đầu diff =", diff)
    for start, end, inc in updates:
        diff[start] += inc
        diff[end + 1] -= inc
        print(f"phép [{start},{end},{inc:+d}]: diff[{start}] += {inc}, diff[{end + 1}] -= {inc} -> {diff}")
    res, running = [], 0
    for i in range(length):
        running += diff[i]
        res.append(running)
        print(f"  i={i}: diff[{i}]={diff[i]:+d}, tổng tiền tố = {running}")
    return res


assert trace_diff(5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]) == [-2, 0, 3, 5, 3]

ban đầu diff = [0, 0, 0, 0, 0, 0]
phép [1,3,+2]: diff[1] += 2, diff[4] -= 2 -> [0, 2, 0, 0, -2, 0]
phép [2,4,+3]: diff[2] += 3, diff[5] -= 3 -> [0, 2, 3, 0, -2, -3]
phép [0,2,-2]: diff[0] += -2, diff[3] -= -2 -> [-2, 2, 3, 2, -2, -3]
  i=0: diff[0]=-2, tổng tiền tố = -2
  i=1: diff[1]=+2, tổng tiền tố = 0
  i=2: diff[2]=+3, tổng tiền tố = 3
  i=3: diff[3]=+2, tổng tiền tố = 5
  i=4: diff[4]=-2, tổng tiền tố = 3


In [19]:
def trace_sweep(length, updates):
    events = {}
    for start, end, inc in updates:
        events[start] = events.get(start, 0) + inc
        events[end + 1] = events.get(end + 1, 0) - inc
    print("sự kiện:", dict(sorted(events.items())))
    res, cur, prev = [0] * length, 0, 0
    for pos in sorted(events):
        stop = min(pos, length)
        if stop > prev:
            res[prev:stop] = [cur] * (stop - prev)
            print(f"  mốc {pos}: điền [{prev},{stop}) bằng {cur} -> {res}")
            prev = stop
        cur += events[pos]
        print(f"  mốc {pos}: cur += {events[pos]:+d} -> cur = {cur}")
    if prev < length:
        res[prev:length] = [cur] * (length - prev)
        print(f"  điền nốt [{prev},{length}) bằng {cur} -> {res}")
    return res


assert trace_sweep(8, [[1, 3, 5], [2, 5, -2], [6, 7, 4]]) == range_add_brute(8, [[1, 3, 5], [2, 5, -2], [6, 7, 4]])

sự kiện: {1: 5, 2: -2, 4: -5, 6: 6, 8: -4}
  mốc 1: điền [0,1) bằng 0 -> [0, 0, 0, 0, 0, 0, 0, 0]
  mốc 1: cur += +5 -> cur = 5
  mốc 2: điền [1,2) bằng 5 -> [0, 5, 0, 0, 0, 0, 0, 0]
  mốc 2: cur += -2 -> cur = 3
  mốc 4: điền [2,4) bằng 3 -> [0, 5, 3, 3, 0, 0, 0, 0]
  mốc 4: cur += -5 -> cur = -2
  mốc 6: điền [4,6) bằng -2 -> [0, 5, 3, 3, -2, -2, 0, 0]
  mốc 6: cur += +6 -> cur = 4
  mốc 8: điền [6,8) bằng 4 -> [0, 5, 3, 3, -2, -2, 4, 4]
  mốc 8: cur += -4 -> cur = 0


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2** (mảng hiệu + `accumulate`): ngắn, `O(n + m)`.

In [20]:
class Solution:
    def getModifiedArray(self, length: int, updates: List[List[int]]) -> List[int]:
        diff = [0] * (length + 1)               # thêm 1 ô cho chỉ số end + 1
        for start, end, inc in updates:
            diff[start] += inc                  # bắt đầu cộng inc từ start
            diff[end + 1] -= inc                # ngừng cộng sau end
        return list(accumulate(diff[:length]))  # tổng tiền tố rồi bỏ ô dự phòng


sol = Solution()
for length, updates, expected in examples:
    print(sol.getModifiedArray(length, updates), "(kỳ vọng", expected, ")")

[-2, 0, 3, 5, 3] (kỳ vọng [-2, 0, 3, 5, 3] )
[0, -4, 2, 2, 2, 4, 4, -4, -4, -4] (kỳ vọng [0, -4, 2, 2, 2, 4, 4, -4, -4, -4] )
[0, 0, 0, 0] (kỳ vọng [0, 0, 0, 0] )
[5] (kỳ vọng [5] )


## 10. Vì sao thuật toán đúng

- **Mô hình mảng hiệu.** Gọi `diff` là mảng mà cuối cùng `arr[i] = diff[0] + ... + diff[i]`. Tổng tiền tố là phép **tuyến tính**: tổng tiền tố của tổng hai mảng bằng tổng hai tổng tiền tố. Vì vậy ta có thể xử lý **từng phép độc lập** rồi cộng các `diff` lại.
- **Một phép là đúng.** Với một phép `[l, r, inc]` dùng riêng, `diff[l] = inc`, `diff[r+1] = -inc`, còn lại `0`. Tổng tiền tố tại `i < l` là `0`; tại `l <= i <= r` là `inc`; tại `i > r` là `inc + (-inc) = 0`. Đúng bằng "cộng `inc` vào đoạn `[l, r]`".
- **Nhiều phép.** Nhờ tính tuyến tính, mảng kết quả là **tổng** của kết quả từng phép — chính là cộng dồn các cập nhật như đề.
- **Ô dự phòng.** `r + 1` có thể bằng `length`; ô này nằm ngoài mảng kết quả nên không ảnh hưởng `arr[0..length-1]`, ta chỉ cần nó **tồn tại** để tránh `IndexError`.
- **Ví dụ không có phép nào.** `diff` toàn 0 ⇒ `arr` toàn 0, đúng đề.

## 11. Phân tích Time / Space Complexity

Gọi `n = length`, `m = len(updates)`.

| Cách | Time | Space (phụ) |
|---|---|---|
| 1. Cộng trực tiếp | `O(n·m)` | `O(1)` |
| 2. Mảng hiệu + tổng tiền tố | `O(n + m)` | `O(n)` |
| 3. Sweep line theo mốc | `O(m log m + n)` | `O(m)` phụ (+ `O(n)` kết quả) |
| 4. Cây Fenwick | `O((n + m) log n)` | `O(n)` |

Với `n = 10^5`, `m = 10^4`: Cách 1 tới `~10^9` bước (**TLE**), Cách 2 chỉ `~1.1 × 10^5` bước. Mọi cách đều phải ghi ra `n` phần tử kết quả nên không thể tốt hơn `O(n)`.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng. Mọi cách trả về **mảng duy nhất** (không có nhiều đáp án đúng) nên so sánh bằng `==`. Ngoài **brute force**, ta thêm một **oracle độc lập** dựa trên định nghĩa: `arr[i] = tổng inc của các phép có start <= i <= end`.

In [21]:
sol = Solution()
impls = {
    "brute": range_add_brute,
    "diff": range_add_diff,
    "sweep": range_add_sweep,
    "fenwick": range_add_fenwick,
    "leetcode": sol.getModifiedArray,
}


def oracle(length, updates):
    """Độc lập: với từng chỉ số i, cộng inc của các phép phủ i (không dùng mảng hiệu)."""
    return [sum(inc for s, e, inc in updates if s <= i <= e) for i in range(length)]


def check_all(length, updates, expected, label):
    for key, fn in impls.items():
        actual = fn(length, [list(u) for u in updates])
        assert actual == expected, (label, key, length, updates, actual, expected)
    assert oracle(length, updates) == expected, (label, "oracle")
    print("PASS", label)


for idx, (length, updates, expected) in enumerate(examples, 1):
    check_all(length, updates, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3
PASS ví dụ 4


### 12.1. Các trường hợp biên

In [22]:
check_all(1, [], [0], "length = 1, không có phép")
check_all(1, [[0, 0, -1000]], [-1000], "length = 1, inc âm tối thiểu")
check_all(3, [[0, 2, 1]], [1, 1, 1], "phủ cả mảng (end = length - 1)")
check_all(3, [[0, 0, 4]], [4, 0, 0], "chỉ phần tử đầu")
check_all(3, [[2, 2, 4]], [0, 0, 4], "chỉ phần tử cuối (end + 1 == length)")
check_all(5, [[1, 3, 2], [1, 3, -2]], [0, 0, 0, 0, 0], "hai phép triệt tiêu")
check_all(4, [[1, 1, 5], [1, 1, 5], [1, 1, 5]], [0, 15, 0, 0], "lặp cùng một phép")
check_all(6, [[0, 5, 1000], [0, 5, 1000]], [2000] * 6, "cộng chồng toàn mảng")
check_all(6, [[0, 2, 3], [3, 5, 4]], [3, 3, 3, 4, 4, 4], "hai đoạn kề nhau không chồng")
check_all(5, [[2, 4, -7], [0, 1, 7]], [7, 7, -7, -7, -7], "đoạn âm và dương")

PASS length = 1, không có phép
PASS length = 1, inc âm tối thiểu
PASS phủ cả mảng (end = length - 1)
PASS chỉ phần tử đầu
PASS chỉ phần tử cuối (end + 1 == length)
PASS hai phép triệt tiêu
PASS lặp cùng một phép
PASS cộng chồng toàn mảng
PASS hai đoạn kề nhau không chồng
PASS đoạn âm và dương


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi hay gặp: (a) **trừ `inc` tại `end`** thay vì `end + 1`; (b) **không lấy tổng tiền tố** (trả về `diff`); (c) **cắt lát lệch một** (`arr[start:end]`). Ta chứng minh các hàm sai này cho kết quả **khác** đúng ở một số input.

In [23]:
def wrong_minus_at_end(length, updates):
    diff = [0] * (length + 1)
    for s, e, inc in updates:
        diff[s] += inc
        diff[e] -= inc                       # SAI: phải là e + 1
    return list(accumulate(diff[:length]))


def wrong_no_prefix(length, updates):
    diff = [0] * (length + 1)
    for s, e, inc in updates:
        diff[s] += inc
        diff[e + 1] -= inc
    return diff[:length]                     # SAI: quên tổng tiền tố


def wrong_slice(length, updates):
    arr = [0] * length
    for s, e, inc in updates:
        for i in range(s, e):                # SAI: bỏ sót phần tử e
            arr[i] += inc
    return arr


L, U = 5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]
for fn in (wrong_minus_at_end, wrong_no_prefix, wrong_slice):
    print(fn.__name__, fn(L, U), "!=", sol.getModifiedArray(L, U))
    assert fn(L, U) != sol.getModifiedArray(L, U)

wrong_minus_at_end [-2, 0, 5, 3, 0] != [-2, 0, 3, 5, 3]
wrong_no_prefix [-2, 2, 3, 2, -2] != [-2, 0, 3, 5, 3]
wrong_slice [-2, 0, 5, 3, 0] != [-2, 0, 3, 5, 3]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Đầu vào hợp lệ theo ràng buộc: `0 <= start <= end < length`, `-1000 <= inc <= 1000`. Ta kiểm tra: (1) **nhiều test nhỏ** so cả 4 cách với oracle độc lập; (2) **test lớn** (`length` tới `10^5`, `10^4` phép) so Cách 2, 3, 4 với nhau; (3) một test **brute force vừa phải** để chắc cả Cách 1 cũng khớp.

In [24]:
rng = random.Random(370)


def random_updates(length, m):
    ups = []
    for _ in range(m):
        s = rng.randint(0, length - 1)
        e = rng.randint(s, length - 1)
        ups.append([s, e, rng.randint(-1000, 1000)])
    return ups


for trial in range(2000):
    length = rng.randint(1, 30)
    updates = random_updates(length, rng.randint(0, 12))
    ref = oracle(length, updates)
    for key, fn in impls.items():
        assert fn(length, [list(u) for u in updates]) == ref, (key, length, updates)
print("PASS 2000 test nhỏ ngẫu nhiên (cả 5 hàm + oracle)")

for trial in range(3):
    length = 100_000
    updates = random_updates(length, 10_000)
    ref = range_add_diff(length, updates)
    assert range_add_sweep(length, updates) == ref
    assert range_add_fenwick(length, updates) == ref
    assert sol.getModifiedArray(length, updates) == ref
print("PASS 3 test lớn: length = 10^5, 10^4 phép (Cách 2, 3, 4)")

length = 3000
updates = random_updates(length, 400)
assert range_add_brute(length, updates) == range_add_diff(length, updates)
print("PASS brute force vừa phải: length = 3000, 400 phép")

PASS 2000 test nhỏ ngẫu nhiên (cả 5 hàm + oracle)


PASS 3 test lớn: length = 10^5, 10^4 phép (Cách 2, 3, 4)
PASS brute force vừa phải: length = 3000, 400 phép


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trừ `inc` tại `end` thay vì `end + 1`

In [25]:
def loi1_tru_tai_end(length, updates):
    diff = [0] * (length + 1)
    for s, e, inc in updates:
        diff[s] += inc
        diff[e] -= inc                # SAI: đoạn đóng, phần tử e vẫn phải được cộng
    return list(accumulate(diff[:length]))


print("sai:", loi1_tru_tai_end(5, [[1, 3, 2]]), "| đúng:", Solution().getModifiedArray(5, [[1, 3, 2]]))

sai: [0, 2, 2, 0, 0] | đúng: [0, 2, 2, 2, 0]


**Vì sao sai:** đoạn `[1, 3]` gồm cả chỉ số `3`. Nếu ta trừ lại `inc` tại `3`, thì phần tử `3` mất phần cộng. Phải trừ **ngay sau** `end`, tức tại `end + 1`. **Cách sửa:** `diff[end + 1] -= inc`.

### Lỗi 2: `diff` chỉ có `length` ô nên `end + 1` tràn chỉ số

In [26]:
def loi2_tran_chi_so(length, updates):
    diff = [0] * length             # SAI: thiếu một ô
    for s, e, inc in updates:
        diff[s] += inc
        diff[e + 1] -= inc          # e + 1 == length gây IndexError
    return list(accumulate(diff))


try:
    loi2_tran_chi_so(3, [[0, 2, 1]])
except IndexError as err:
    print("IndexError:", err)
print("đúng:", Solution().getModifiedArray(3, [[0, 2, 1]]))

IndexError: list index out of range
đúng: [1, 1, 1]


**Vì sao sai:** khi `end = length - 1` thì `end + 1 = length`, ngoài phạm vi list `length` ô. **Cách sửa:** cấp `length + 1` ô cho `diff`, hoặc chỉ trừ khi `end + 1 < length`.

### Lỗi 3: Quên lấy tổng tiền tố

In [27]:
def loi3_quen_prefix(length, updates):
    diff = [0] * (length + 1)
    for s, e, inc in updates:
        diff[s] += inc
        diff[e + 1] -= inc
    return diff[:length]            # SAI: đây mới là mảng hiệu, chưa phải kết quả


print("sai:", loi3_quen_prefix(5, [[1, 3, 2]]), "| đúng:", Solution().getModifiedArray(5, [[1, 3, 2]]))

sai: [0, 2, 0, 0, -2] | đúng: [0, 2, 2, 2, 0]


**Vì sao sai:** `diff` chỉ ghi **chỗ thay đổi** (`[0, 2, 0, 0, -2]`), chưa lan ra cả đoạn. Phải cộng dồn (`accumulate`) mới được `[0, 2, 2, 2, 0]`. **Cách sửa:** `list(accumulate(diff[:length]))`.

### Lỗi 4: Cắt lát lệch một

In [28]:
def loi4_lech_mot(length, updates):
    arr = [0] * length
    for s, e, inc in updates:
        for i in range(s, e):       # SAI: range không lấy e
            arr[i] += inc
    return arr


print("sai:", loi4_lech_mot(5, [[1, 3, 2]]), "| đúng:", Solution().getModifiedArray(5, [[1, 3, 2]]))

sai: [0, 2, 2, 0, 0] | đúng: [0, 2, 2, 2, 0]


**Vì sao sai:** `range(s, e)` dừng trước `e`, trong khi đề tính cả hai đầu. **Cách sửa:** `range(s, e + 1)`.

### Lỗi 5: Gán `arr[s:e+1] += inc` trên list

In [29]:
arr = [0] * 5
try:
    arr[1:4] += 2                   # SAI: list không hỗ trợ cộng số vào cả lát cắt (khác numpy)
except TypeError as err:
    print("TypeError:", err)
arr[1:4] = [x + 2 for x in arr[1:4]]     # muốn làm kiểu này phải viết tường minh (vẫn là O(độ dài đoạn))
print(arr)

TypeError: 'int' object is not iterable
[0, 2, 2, 2, 0]


**Vì sao sai:** `list += số` không hợp lệ (`+=` với list đòi một iterable). Với list thuần Python không có "cộng cả lát cắt trong `O(1)`". **Cách sửa:** dùng mảng hiệu để mỗi phép chỉ `O(1)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Nhiều phép cộng đoạn, hỏi mảng cuối cùng**" (chỉ hỏi sau khi áp dụng hết) ⇒ **mảng hiệu + tổng tiền tố**.
- Công thức nhớ: `diff[l] += v`, `diff[r+1] -= v`; kết quả = `accumulate(diff)`.
- Luôn cấp `n + 1` ô (hoặc kiểm tra `r + 1 < n`) để tránh `IndexError`.
- Nếu đề **1-indexed** (như Corporate Flight Bookings) đổi chỉ số về `0` hoặc cấp thêm một ô và nhớ bỏ ô `0`.
- Khi cần **vừa cập nhật đoạn vừa truy vấn xen kẽ**, dùng **Fenwick** (cập nhật đoạn - truy vấn điểm) hoặc **segment tree** với lazy propagation.
- Khi **tọa độ rất lớn nhưng ít phép** (không thể cấp mảng), dùng **sweep line** trên các mốc đã sắp xếp.
- Kiểm tra bằng **oracle**: `arr[i] = tổng inc của các phép phủ i` — độc lập với mảng hiệu.

## 15. Biến thể của bài

1. **Corporate Flight Bookings (1109)** — cùng ý, chỉ số bắt đầu từ `1`.
2. **Car Pooling (1094)** — kiểm tra sức chứa tại mọi thời điểm (đường đi `0..1000`).
3. **Range Addition II (598)** — chỉ có các phép cộng `1` vào hình chữ nhật góc trên trái; đếm số ô lớn nhất.
4. **Range Sum / Range Min Query** sau nhiều cập nhật (Fenwick / segment tree).
5. **Mảng hiệu 2D** — cộng hình chữ nhật con (4 điểm đánh dấu: `+`, `-`, `-`, `+`).
6. **Zero Array Transformation I (3355)** — kiểm tra mảng có về `0` được không sau các lần giảm đoạn.

In [30]:
def corporate_flight_bookings(bookings, n):
    """1109: bookings[i] = [first, last, seats], chỉ số 1..n -> trả về mảng độ dài n."""
    diff = [0] * (n + 2)
    for first, last, seats in bookings:
        diff[first] += seats
        diff[last + 1] -= seats
    return list(accumulate(diff))[1:n + 1]


def car_pooling(trips, capacity):
    """1094: trips[i] = [người, lên, xuống]; người xuống trước khi người mới lên cùng điểm."""
    diff = [0] * 1002
    for people, start, end in trips:
        diff[start] += people
        diff[end] -= people             # xuống tại end: tại điểm end đã không còn tính người này
    return all(x <= capacity for x in accumulate(diff))


print(corporate_flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5))   # [10, 55, 45, 25, 25]
print(car_pooling([[2, 1, 5], [3, 3, 7]], 4), car_pooling([[2, 1, 5], [3, 3, 7]], 5))   # False True

[10, 55, 45, 25, 25]
False True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cộng trực tiếp | vòng `for` cho từng phần tử của đoạn | `O(n·m)` | `O(1)` | `n·m` nhỏ; làm **oracle** |
| 2. Mảng hiệu + tổng tiền tố | `+inc` tại `start`, `-inc` tại `end+1` | `O(n + m)` | `O(n)` | **nên nộp** |
| 3. Sweep line | sắp xếp mốc, điền theo đoạn | `O(m log m + n)` | `O(m)` phụ | tọa độ lớn, ít mốc |
| 4. Fenwick | cập nhật điểm trên mảng hiệu, truy vấn tổng tiền tố | `O((n+m) log n)` | `O(n)` | cập nhật và truy vấn xen kẽ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(accumulate([1, 2, 3, 4]))`, `list(range(2, 5))`, `Solution().getModifiedArray(3, [[0, 1, 5]])`, và `Solution().getModifiedArray(2, [])`.

**Bài 2 (tính tay).** Với `length = 6`, `updates = [[0, 2, 4], [2, 5, -1], [4, 4, 10]]`: viết mảng `diff` (7 ô) sau cả ba phép, rồi mảng kết quả.

**Bài 3 (điền TODO).** Hoàn thành `range_add_todo(length, updates)` theo Cách 2.

In [31]:
def range_add_todo(length, updates):
    # TODO 1: diff = [0] * (length + 1)
    # TODO 2: với mỗi (start, end, inc): diff[start] += inc; diff[end + 1] -= inc
    # TODO 3: cộng dồn diff[0..length-1] và trả về mảng kết quả
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `range_add_fenwick2(length, updates)` hoặc `range_add_sweep2(length, updates)` **không nhìn code mẫu**, rồi tự kiểm tra bằng `assert` so với `Solution().getModifiedArray` trên các ví dụ.

**Bài 5 (biến thể / mini challenge).** (a) Viết `flight_bookings(bookings, n)` cho **1109** (chỉ số bắt đầu từ `1`). (b) Viết `max_after_updates(length, updates)` trả về **giá trị lớn nhất** trong mảng cuối cùng mà **không** tạo toàn bộ `arr` (chỉ duyệt các mốc đã sắp xếp). (c) Ý tưởng cho phiên bản **2D**: cộng `inc` vào hình chữ nhật `(r1, c1)` đến `(r2, c2)` cần đánh dấu ở những ô nào?

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 3, 6, 10]`, `[2, 3, 4]`, `[5, 5, 0]`, `[0, 0]`. Bài 3: chép lại ba bước của Cách 2; `accumulate` hoặc biến `running`.
- **Mức 2 (vừa):** Bài 2: `[0,2,4]` cho `diff[0] += 4`, `diff[3] -= 4`; `[2,5,-1]` cho `diff[2] -= 1`, `diff[6] += 1`; `[4,4,10]` cho `diff[4] += 10`, `diff[5] -= 10`. Gộp lại ta được `diff` gồm 7 ô, rồi cộng dồn 6 ô đầu. Bài 4: bắt đầu từ ý "chỉ ghi chỗ thay đổi".
- **Mức 3 (sâu):** Bài 5(a): dùng `diff` dài `n + 2`, bỏ ô `0`. 5(b): sắp xếp mốc, duy trì `cur`, giá trị lớn nhất cần xét **sau** mỗi mốc (và nhớ `max` ban đầu có thể là `0` nếu còn phần tử chưa bị phủ). 5(c): `+inc` tại `(r1, c1)`, `-inc` tại `(r1, c2+1)`, `-inc` tại `(r2+1, c1)`, `+inc` tại `(r2+1, c2+1)`; sau đó lấy tổng tiền tố theo hàng rồi theo cột.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert list(accumulate([1, 2, 3, 4])) == [1, 3, 6, 10]
assert list(range(2, 5)) == [2, 3, 4]
assert Solution().getModifiedArray(3, [[0, 1, 5]]) == [5, 5, 0]
assert Solution().getModifiedArray(2, []) == [0, 0]

# Bài 2
length, updates = 6, [[0, 2, 4], [2, 5, -1], [4, 4, 10]]
diff = [0] * 7
for s, e, inc in updates:
    diff[s] += inc
    diff[e + 1] -= inc
assert diff == [4, 0, -1, -4, 10, -10, 1]
assert Solution().getModifiedArray(length, updates) == [4, 4, 3, -1, 9, -1]


# Bài 3
def range_add_todo_ans(length, updates):
    diff = [0] * (length + 1)
    for start, end, inc in updates:
        diff[start] += inc
        diff[end + 1] -= inc
    res, running = [], 0
    for i in range(length):
        running += diff[i]
        res.append(running)
    return res


for length, updates, expected in examples:
    assert range_add_todo_ans(length, updates) == expected


# Bài 4 (bản sweep viết lại)
def range_add_sweep2(length, updates):
    ev = {}
    for s, e, inc in updates:
        ev[s] = ev.get(s, 0) + inc
        ev[e + 1] = ev.get(e + 1, 0) - inc
    res, cur, prev = [0] * length, 0, 0
    for pos in sorted(ev):
        stop = min(pos, length)
        for i in range(prev, stop):
            res[i] = cur
        prev = max(prev, stop)
        cur += ev[pos]
    for i in range(prev, length):
        res[i] = cur
    return res


for length, updates, expected in examples:
    assert range_add_sweep2(length, updates) == expected


# Bài 5(a)
def flight_bookings(bookings, n):
    diff = [0] * (n + 2)
    for first, last, seats in bookings:
        diff[first] += seats
        diff[last + 1] -= seats
    return list(accumulate(diff))[1:n + 1]


assert flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5) == [10, 55, 45, 25, 25]


# Bài 5(b)
def max_after_updates(length, updates):
    ev = {}
    for s, e, inc in updates:
        ev[s] = ev.get(s, 0) + inc
        ev[e + 1] = ev.get(e + 1, 0) - inc
    best, cur, prev = None, 0, 0
    for pos in sorted(ev):
        if pos > prev and prev < length:          # đoạn [prev, min(pos, length)) có giá trị cur
            best = cur if best is None else max(best, cur)
        cur += ev[pos]
        prev = pos
    if prev < length:                             # đoạn cuối [prev, length)
        best = cur if best is None else max(best, cur)
    return best


rng2 = random.Random(1)
for _ in range(500):
    L = rng2.randint(1, 20)
    ups = []
    for _ in range(rng2.randint(0, 6)):
        s = rng2.randint(0, L - 1)
        e = rng2.randint(s, L - 1)
        ups.append([s, e, rng2.randint(-5, 5)])
    assert max_after_updates(L, ups) == max(Solution().getModifiedArray(L, ups)), (L, ups)
print("Đáp án bài 1-5 đều đúng")

Đáp án bài 1-5 đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nhiều phép cộng đoạn, hỏi mảng cuối cùng)

```python
class Solution:
    def getModifiedArray(self, length: int, updates: List[List[int]]) -> List[int]:
        diff = [0] * (length + 1)
        for start, end, inc in updates:
            diff[start] += inc
            diff[end + 1] -= inc
        return list(accumulate(diff[:length]))
```

**Checklist**

- [ ] `diff` có `length + 1` ô (hoặc đã kiểm tra `end + 1 < length`)?
- [ ] Trừ `inc` tại **`end + 1`**, không phải `end`?
- [ ] Đã lấy **tổng tiền tố** (`accumulate`) trước khi trả về?
- [ ] Đã bỏ ô dự phòng cuối (`diff[:length]`)?
- [ ] Trường hợp `updates` rỗng trả về toàn `0`?
- [ ] Đã thử `end = length - 1` và `length = 1`?
- [ ] Nếu đề đổi sang 1-indexed, đã đổi chỉ số cho khớp?